# Tidy Data & Combine Tables

## melt() method

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

In [3]:
table1 = pd.DataFrame(
    {
        'candidate':['Hillary Clinton', 'Donald Trump', 'Gary Johnson', 'Jill Stein'],
        'CA': [5931283, 5931283, 308392, 308392],
        'FL': [4485745, 4605515, 206007, 64019]
    }
)

table1

,candidate,CA,FL
0,Hillary Clinton,5931283,4485745
1,Donald Trump,5931283,4605515
2,Gary Johnson,308392,206007
3,Jill Stein,308392,64019


In [4]:
melted = pd.melt(table1, id_vars= 'candidate', value_vars= ['CA', 'FL'], 
        var_name= 'states', value_name='votes')

melted.head()

,candidate,states,votes
0,Hillary Clinton,CA,5931283
1,Donald Trump,CA,5931283
2,Gary Johnson,CA,308392
3,Jill Stein,CA,308392
4,Hillary Clinton,FL,4485745


## pivot() method

In [5]:
pivotTable = melted.pivot(
    index=  'candidate',
    columns = 'states',
    values= 'votes'
).reset_index()

pivotTable

states,candidate,CA,FL
0,Donald Trump,5931283,4605515
1,Gary Johnson,308392,206007
2,Hillary Clinton,5931283,4485745
3,Jill Stein,308392,64019


--------

## Splitting Columns

In [6]:
table2 = pd.DataFrame({
    'candidate':['Hillary Clinton', 'Donald Trump', 'Gary Johnson'],
    'state' : 'CA',
    'proportion': ['5931283/9631972', '3184721/9631972', '308392/9631972'] 
})

table2

,candidate,state,proportion
0,Hillary Clinton,CA,5931283/9631972
1,Donald Trump,CA,3184721/9631972
2,Gary Johnson,CA,308392/9631972


In [7]:
table2[['votes', 'total_votes']] = table2['proportion'].str.split('/', expand= True).astype(int)
table2

,candidate,state,proportion,votes,total_votes
0,Hillary Clinton,CA,5931283/9631972,5931283,9631972
1,Donald Trump,CA,3184721/9631972,3184721,9631972
2,Gary Johnson,CA,308392/9631972,308392,9631972


-----

# Concatenating Tables

In [8]:
import glob

In [25]:
files = glob.glob('/Users/ulasuregen/repos/Data-Analysis-and-Visualization-in-Python/Datasets/extdata2/cov_concatenate/*.csv')
print(files[:5])

['/Users/ulasuregen/repos/Data-Analysis-and-Visualization-in-Python/Datasets/extdata2/cov_concatenate/covid_cases_01_03_2020.csv', '/Users/ulasuregen/repos/Data-Analysis-and-Visualization-in-Python/Datasets/extdata2/cov_concatenate/covid_cases_08_03_2020.csv', '/Users/ulasuregen/repos/Data-Analysis-and-Visualization-in-Python/Datasets/extdata2/cov_concatenate/covid_cases_26_03_2020.csv', '/Users/ulasuregen/repos/Data-Analysis-and-Visualization-in-Python/Datasets/extdata2/cov_concatenate/covid_cases_14_03_2020.csv', '/Users/ulasuregen/repos/Data-Analysis-and-Visualization-in-Python/Datasets/extdata2/cov_concatenate/covid_cases_23_03_2020.csv']


In [27]:
df = pd.read_csv(files[0])
df

,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000
0,54,0,Germany,DE,DEU,83019213,Europe,0.115636
1,240,8,Italy,IT,ITA,60359546,Europe,1.863831


In [35]:
from pathlib import Path

dfs = []

for f in files:
    name = Path(f).name
    df = pd.read_csv(f).assign(filename = name)
    dfs.append(df)

df_concat = pd.concat(dfs, ignore_index= True)
df_concat.head(3)

,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000,filename
0,54,0,Germany,DE,DEU,83019213,Europe,0.115636,covid_cases_01_03_2020.csv
1,240,8,Italy,IT,ITA,60359546,Europe,1.863831,covid_cases_01_03_2020.csv
2,163,0,Germany,DE,DEU,83019213,Europe,1.002178,covid_cases_08_03_2020.csv


Alternatively we can achieve the same result in a single line using a Python list comprehension.

In [37]:
df_concat = pd.concat([
    pd.read_csv(f).assign(filename = Path(f).name)
    for f in files
], ignore_index= True)

df_concat.head(3)

,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000,filename
0,54,0,Germany,DE,DEU,83019213,Europe,0.115636,covid_cases_01_03_2020.csv
1,240,8,Italy,IT,ITA,60359546,Europe,1.863831,covid_cases_01_03_2020.csv
2,163,0,Germany,DE,DEU,83019213,Europe,1.002178,covid_cases_08_03_2020.csv


-----

# Merging Tables

In [50]:
df1 = pd.DataFrame({
'p_id': ['G008', 'F027', 'L051'],
'value': np.random.normal(size=3)
})
df1

,p_id,value
0,G008,1.062671
1,F027,1.483051
2,L051,-0.688556


In [51]:
df2 = pd.DataFrame({
'p_id': ['G008', 'F027', 'U093'],
'country': ['Germany', 'France', 'USA']
})
df2

,p_id,country
0,G008,Germany
1,F027,France
2,U093,USA


-----
## Inner Join

In [52]:
m = df1.merge(df2, on = 'p_id', how = 'inner')
m

,p_id,value,country
0,G008,1.062671,Germany
1,F027,1.483051,France


------
## Outer(Full) Join

In [53]:
m = df1.merge(df2, on = 'p_id', how = 'outer')
m

,p_id,value,country
0,F027,1.483051,France
1,G008,1.062671,Germany
2,L051,-0.688556,NaN
3,U093,NaN,USA


------
## Left Join

In [54]:
m = df1.merge(df2, on = 'p_id', how = 'left')
m

,p_id,value,country
0,G008,1.062671,Germany
1,F027,1.483051,France
2,L051,-0.688556,NaN


-------
## Right Join

In [55]:
m = df1.merge(df2, on = 'p_id', how = 'right')
m

,p_id,value,country
0,G008,1.062671,Germany
1,F027,1.483051,France
2,U093,NaN,USA


---------
## Merging by more than one column

In [57]:
df1 = pd.DataFrame({
'firstname': ['Alice', 'Alice', 'Bob'],
'lastname': ['Coop', 'Smith', 'Smith'],
'x': [1, 2, 3]
})
df1

,firstname,lastname,x
0,Alice,Coop,1
1,Alice,Smith,2
2,Bob,Smith,3


In [59]:
df2 = pd.DataFrame({
'firstname': ['Alice', 'Bob', 'Bob'],
'lastname': ['Coop', 'Marley','Smith'],
'y': list('ABC')
})
df2

,firstname,lastname,y
0,Alice,Coop,A
1,Bob,Marley,B
2,Bob,Smith,C


In [60]:
df1.merge(df2, on = ['firstname', 'lastname'])

,firstname,lastname,x,y
0,Alice,Coop,1,A
1,Bob,Smith,3,C


---------